<a href="https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sgln24/flyrank-ml-internship-w1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

**One row**

One row represents one content item on one reporting day.

**Time window**

Tables:

fact_daily
dim_content

Time Window:

March 2026 (mid-panel month)

**Prediction target**

Predict total page impressions for a content item during the selected month.

Excluded:
- Label-derived features (to avoid leakage)
- June 2026 data (reserved as sealed test month)

In [72]:
# This cell is for CODE (numbers, a query, a check).
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 1
""").df()

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    COUNT(DISTINCT report_date) AS unique_days
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date)=DATE '2026-03-01'
""").df()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content,unique_days
0,9841378,331437,31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

- avg_position
- clicks
- active_days
- avg_ctr
- client_has_gsc

**Label**

- impressions

- Context

- client_hash_id

- content_hash_id

- report_date

**Excluded**

- impressions-derived features
June 2026 data

**Reason:**

- Features derived from impressions would introduce label leakage because they reveal the prediction target.

- June 2026 is deliberately excluded because it is treated as the sealed final test month, following the assignment instructions.

In [73]:
# This cell is for CODE (numbers, a query, a check).

con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This notebook uses the March 2026 slice from the fact_daily table as the development (mid-panel) month. June 2026 is treated as the final test month and is not used for development.

In [74]:
# This cell is for CODE (numbers, a query, a check).
duplicates = con.sql(f"""
SELECT
    content_hash_id,
    report_date,
    COUNT(*) AS rows_per_key
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
GROUP BY content_hash_id, report_date
HAVING COUNT(*) > 1
""").df()

print("Duplicate keys:", len(duplicates))
duplicates.head()
# Query 2 (Row count + Date span for mid-panel month)

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

# Query 3 (Availability)

con.sql(f"""
SELECT
    COUNT(*) AS available_content
FROM {TABLES['dim_content']}
WHERE is_published IS TRUE
  AND is_deleted IS FALSE
""").df()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate keys: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_content
0,411540


## Feature frame

The following feature frame contains five features for the Ranking Signal Analysis lane.

These features are selected because they are available before the prediction target is observed and therefore can be safely used for modeling.

In [75]:
#Using a subset (LIMIT 10000) for quick experimentation.
feature_frame = con.sql(f"""
SELECT
    content_hash_id,
    AVG(gsc_avg_position) AS avg_position,
    SUM(gsc_clicks) AS clicks,
    COUNT(DISTINCT report_date) AS active_days,
    AVG(
        CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        END
    ) AS avg_ctr,
    MAX(client_has_gsc) AS client_has_gsc,
    SUM(gsc_impressions) AS impressions
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date)=DATE '2026-03-01'
GROUP BY content_hash_id
LIMIT 10000
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,avg_position,clicks,active_days,avg_ctr,client_has_gsc,impressions
0,content_d0dff76c889de68f,5.147402,0.0,31,0.000000,True,181.0
1,content_67741cce996cfafa,4.828125,1.0,31,0.015625,True,46.0
2,content_2e6360ad20fd7107,5.145765,1.0,31,0.001613,True,899.0
3,content_ac8663da7484669a,4.909314,0.0,31,0.000000,True,34.0
4,content_65c50dfe9d87a585,6.969536,0.0,31,0.000000,True,3108.0


### Feature availability

| Feature        | Available? | Why                                                         |
| -------------- | ---------- | ----------------------------------------------------------- |
| avg_position   | Yes        | Historical ranking information available before prediction. |
| clicks         | Yes        | Historical clicks observed before prediction.               |
| active_days    | Yes        | Known before prediction.                                    |
| avg_ctr        | Yes        | Historical click-through rate available before prediction.  |
| client_has_gsc | Yes        | Indicates whether GSC data exists before prediction.        |





## Label leakage experiment

This experiment intentionally introduces a label-derived feature to demonstrate how leakage can artificially inflate model performance.

After observing the effect, the leaked feature is removed.

In [76]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

df = feature_frame.dropna()

X = df[
[
    "avg_position",
    "clicks",
    "active_days",
    "avg_ctr",
    "client_has_gsc",
]
]
y = df["impressions"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest R² =", r2_score(y_test, pred))

Honest R² = 0.8962128375174245


In [77]:
X_leak = df[
[
    "avg_position",
    "clicks",
    "active_days",
    "avg_ctr",
    "client_has_gsc",
    "impressions",
]
]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42,
)

leak_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)

leak_model.fit(X_train, y_train)

pred = leak_model.predict(X_test)

print("Leaked R² =", r2_score(y_test, pred))

Leaked R² = 0.9998523810615947


### Leakage observation

Adding impressions as an input feature produced an unrealistically high score because it directly contains the prediction target (label leakage).

This is an example of **label leakage**.

The leaked feature was removed after the demonstration.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits
- This notebook analyzes only the March 2026 slice of the data.
- June 2026 is intentionally excluded because it is reserved as the sealed test month.
- The data measures observed Search Console performance, not causal effects.
- Different clients may have different amounts of historical data available.

In [78]:
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day,
    COUNT(DISTINCT report_date) AS total_days
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061.0,413966.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.